In [4]:
from itertools import groupby
import torch
import transformers 
from transformers import Wav2Vec2FeatureExtractor, Wav2Vec2Tokenizer, Wav2Vec2Processor, Wav2Vec2ForCTC,  Wav2Vec2CTCTokenizer, Wav2Vec2PhonemeCTCTokenizer
import librosa
import argparse
#import eelbrain as eel
import eng_to_ipa as ipa
import pandas as pd
import numpy as np
from phonemizer import phonemize
import torchaudio
import os
import re

audio_filepath ="C:/Users/stell/desktop/cans_lab/Female_Single_Speakers/F01_single105.wav"

arpabet2ipa = {
    'AA':'ɑ',
    'AA0':'a',
    'AA1':'ɐ',
    'AE':'æ',
    'AH':'ʌ',
    'AH0':'ə',
    'AO':'ɔ',
    'AW':'aʊ',
    'AY':'aɪ',
    'EH':'ɛ',
    'ER':'ɝ',
    'ER0':'ɚ',
    'ER1': 'ər',
    'EY':'eɪ',
    'IH':'ɪ',
    'IH0':'ɨ',
    'IY':'i',
    'O':'o',
    'OW':'oʊ',
    'OY':'ɔɪ',
    'OY0':'oɪ',
    'UH':'ʊ',
    'UW':'u',
    'B':'b',
    'CH':'tʃ',
    'CH0':'ʧ',
    'D':'d',
    'DH':'ð',
    'EL':'l̩ ',
    'EM':'m̩',
    'EN':'n̩',
    'F':'f',
    'G':'g',
    'G0':'ɡ',
    'HH':'h',
    'JH':'dʒ',
    'JH0':'ʤ',
    'K':'k',
    'L':'l',
    'M':'m',
    'N':'n',
    'NG':'ŋ',
    'P':'p',
    'Q':'ʔ',
    'R':'ɹ',
    'RR':'r',
    'S':'s',
    'SH':'ʃ',
    'T':'t',
    'T0':'ɾ',
    'TH':'θ',
    'V':'v',
    'W':'w',
    'WH':'ʍ',
    'Y':'j',
    'Z':'z',
    'ZH':'ʒ'
}

ipa2arpabet = dict([(value, key) for key, value in arpabet2ipa.items()])
arpabet = list(arpabet2ipa.keys())

# corrections were determined out of band in a separate script; phonemes with
# less than 20 instances were set to 0 bias due to lack of data.
correction = {
    'N/A': 0.0, # phoneme not recognized, default to word onset time (0 bias)
    'a': 0.0,
    'aɪ': 0.0,
    'aʊ': 0.0,
    'b': 0.0273137254902025,
    'd': 0.010840996877361064,
    'dʒ': 0.0,
    'eɪ': 0.0,
    'f': 0.073921307102367,
    'h': 0.032686324714017445,
    'i': 0.08094752107244707,
    'j': 0.0,
    'k': 0.007710410189417871,
    'l': 0.06733196046128143,
    'm': 0.06112231113256161,
    'n': 0.05480637254902376,
    'o': 0.0,
    'oʊ': 0.0,
    'oː': 0.0,
    'p': 0.00361545986718248,
    's': 0.09089372726133593,
    't': 0.011572554518172629,
    'tʃ': 0.0,
    'u': 0.0,
    'v': 0.0,
    'w': 0.04421147058823749,
    'æ': 0.04221674054039681,
    'ð': 0.012261764496436456,
    'ŋ': 0.0,
    'ɐ': 0.029734671333123686,
    'ɑ': 0.0,
    'ɔ': 0.0,
    'ə': 0.040999408453004094,
    'ɚ': 0.0,
    'ɛ': 0.0,
    'ɡ': 0.04172284737497023,
    'ɪ': 0.03661764705882575,
    'ɹ': 0.056741440234150176,
    'ɾ': 0.0,
    'ʃ': 0.0,
    'ʊ': 0.0,
    'ʌ': 0.03150195552649926,
    'θ': 0.0
}

In [5]:
argParser = argparse.ArgumentParser()
argParser.add_argument("-f", "--file", type=str, help="wav file to process")
argParser.add_argument("-d", "--dest", type=str, help="path to save predictor")
args = argParser.parse_args()

################################################################################
# load model and audio and run audio through model
################################################################################

# load pretrained models
model_name = 'facebook/wav2vec2-large-960h-lv60-self'
#tokenizer = Wav2Vec2Tokenizer.from_pretrained(model_name)
#feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)
model = Wav2Vec2ForCTC.from_pretrained(model_name)
processor = Wav2Vec2Processor.from_pretrained(model_name)


phoneme_model_name = 'facebook/wav2vec2-xlsr-53-espeak-cv-ft'
#phoneme_tokenizer = Wav2Vec2Tokenizer.from_pretrained(phoneme_model_name)
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(phoneme_model_name)
phoneme_model = Wav2Vec2ForCTC.from_pretrained(phoneme_model_name)
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained(phoneme_model_name)
phoneme_processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer = tokenizer)

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/wav2vec2-large-960h-lv60-self and are newly initialized: ['wav2vec2.masked_spec_embed']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fetching 1 files: 100%|███████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<?, ?it/s]
The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'Wav2Vec2PhonemeCTCTokenizer'. 
The class this function is called from is 'Wav2Vec2CTCTokenizer'.


In [6]:
#####################--------------------------------------------
def decoding_to_timings(transcription, predicted_ids, input_values, processor, 
                        sr):
    symbols = [s for s in transcription.split(' ') if len(s) > 0]
    predicted_ids = predicted_ids[0].tolist()
    duration_sec = input_values.shape[1] / sr


    ids_s_time = [(i / len(predicted_ids) * duration_sec, _id) for i, _id in \
                  enumerate(predicted_ids)]

    # now split the ids into groups of ids where each group represents a symbol
    if isinstance(processor.tokenizer, transformers.models.wav2vec2_phoneme.\
                  tokenization_wav2vec2_phoneme.Wav2Vec2PhonemeCTCTokenizer): 
        split_ids_s_time = [list(group) for k, group
                        in groupby(ids_s_time, lambda x: x[1])
                        if k != phoneme_processor.tokenizer.pad_token_id]
    elif isinstance(processor.tokenizer, transformers.models.wav2vec2.\
                    tokenization_wav2vec2.Wav2Vec2CTCTokenizer):
        ids_s_time = [i for i in ids_s_time \
                      if i[1] != processor.tokenizer.pad_token_id]
        split_ids_s_time = [list(group) for k, group in groupby(ids_s_time, \
                lambda x: x[1] == processor.tokenizer.word_delimiter_token_id)
                if not k]
    else:
        raise Exception(f"tokenizer {type(processor.tokenizer)} not supported")

    # make sure that there are the same number of id-groups as words. 
    # otherwise something is wrong
    assert len(split_ids_s_time) == len(symbols)  

    symbol_start_times = []
    symbol_end_times = []
    for cur_ids_s_time, cur_symbol in zip(split_ids_s_time, symbols):
        _times = [_time for _time, _id in cur_ids_s_time]
        symbol_start_times.append(min(_times))
        symbol_end_times.append(max(_times))

     #return symbols, symbol_start_times, symbol_end_times

    #NEW CHANGE

    word_data = []

    for w, s, e in zip(symbols, symbol_start_times, symbol_end_times):
        word_data.append({
            "word": w,
            "start": s,
            "end": e
        })
    
    df = pd.DataFrame(word_data)

    return df

    #END OF CHANGE
   

In [21]:
#Here split it

# load audio
#audio_filepath = "C:/Users/stell/Downloads/pass_mix_F0.wav"
#audio_filepath  = "C:/Users/stell/Desktop/CANS_LAB/Female_Single_Speakers/FO1_single105.wav"

og_waveform, sr = torchaudio.load(audio_filepath)

output_folder = "Wav2Vec_Shifted"
audio_folder = "Female_Single_Speakers"


In [8]:
def shift_audio(og_waveform, shift_ms, sr):
    #print(sr)
    #print(shift_ms)
    shift_samples = int(sr * (shift_ms / 1000.0))
    #print(shift_samples)

    if og_waveform.dim() == 1:
        og_waveform = og_waveform.unsqueeze(0)
        
    if shift_samples > 0:
        shifted = torch.cat([torch.zeros(1, shift_samples), og_waveform[:, :-shift_samples]], dim=1)
    elif shift_samples < 0:
        shifted = torch.cat([og_waveform[:, -shift_samples:], torch.zeros(1, -shift_samples)], dim=1)
    else:
        shifted = og_waveform
    return shifted

In [31]:
#ORIGINAL NO SHIFTED
df_noshift  = decoding_to_timings(transcription, \
        predicted_ids, input_values, processor, target_sample_rate)
rounded_df = df_noshift.round(decimals=3)
print(df_noshift.head())
rounded_df.to_csv(f"{output_folder}/Wav2vec_OG.csv", index=False)


    word     start       end
0     in  0.040009  0.100023
1    the  0.180041  0.240055
2  bosom  0.320073  0.640146
3     of  0.780178  0.820187
4    one  0.940214  1.020233


In [ ]:
print(shifted_audio.detach().cpu().numpy().shape)
print(waveform.detach().cpu().numpy().shape)

In [13]:
def process_shift( og_waveform, ms, sr, output_folder, processor, model, decoding_to_timings):


    #assert (not np.all(shifted_audio.detach().cpu().numpy() == og_waveform.detach().cpu().numpy().flatten()))
    shifted_audio = shift_audio(og_waveform, ms, sr)
    
     # Create file path for the shifted version
    base_name = f"{output_folder}/audio_shift_{ms}ms.wav"

    # Save the shifted version
    torchaudio.save(base_name, shifted_audio, sr)

    #Now inputing new audio
    speech, sample_rate = librosa.load(base_name)

    
    # pretrained models need 16kHz, final timings are in clock times, so resampling
    # outputs is not necessary
    target_sample_rate = 16000
    speech = librosa.resample(speech, orig_sr=sample_rate, 
                              target_sr=target_sample_rate)
    
    # run speech through processors
    a = processor(speech, sampling_rate=target_sample_rate, return_tensors="pt")
    b = phoneme_processor(speech, sampling_rate=target_sample_rate, 
                          return_tensors="pt")
    
    input_values = a.input_values
    p_input_values = b.input_values
    
    with torch.no_grad():
        logits = model(input_values).logits
        p_logits = phoneme_model(p_input_values).logits
    
    # decode model outputs
    predicted_ids = torch.argmax(logits, dim=-1)
    p_predicted_ids = torch.argmax(p_logits, dim=-1)
    
    transcription = processor.decode(predicted_ids[0]).lower()
    phonemes = phoneme_processor.decode(p_predicted_ids[0]).lower()
    #phonemes = phoneme_processor.batch_decode(p_predicted_ids)

    df  = decoding_to_timings(transcription, \
        predicted_ids, input_values, processor, target_sample_rate)
    
    round_df = df.round(decimals = 3)

    round_df.rename(columns={"Start": f"Start_{ms}", "End": f"End_{ms}"})

    #csv_path = f"{output_folder}/audio_shift_{ms}ms_timestamps.csv"
    #round_df.to_csv(csv_path, index=False)

    return round_df

In [27]:
#shift_ms = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
shift_ms = list(range(-20,21,4))

print(shift_ms)

[-20, -16, -12, -8, -4, 0, 4, 8, 12, 16, 20]


In [17]:
print("hello")

hello


In [29]:
shifted_dfs = {}
for wav in os.listdir(audio_folder):

    if wav.endswith(".wav"):

        #Get the name without the extension
        base = os.path.splitext(wav)[0]
        #Split each word in the name
        speaker_type = base.split("_")[0]

        #Get path of each audio file
        audio_path = os.path.join(audio_folder, wav)

        og_waveform, sr = torchaudio.load(audio_path)

        speaker_dir = os.path.join(output_folder, speaker_type)
        os.makedirs(speaker_dir, exist_ok=True)
        
        for ms in shift_ms:
            shifted_audio = shift_audio(og_waveform, ms, sr)
        
            #print(shifted_audio.detach().cpu().numpy().shape)
        
        
            shifted_df = process_shift(
                shifted_audio,
                ms,
                sr,
                output_folder,
                processor,
                model,
                decoding_to_timings)

            shifted_df.to_csv(
            os.path.join(speaker_dir, f"{speaker_type}_{ms}ms.csv"),
            index=False)
            
            # store in dict with a unique key
            shifted_dfs[f"Shift_{speaker_type}_{ms}ms"] = shifted_df
        
            print(f"Finished {speaker_type}_{ms}ms")

Finished F01_-20ms
Finished F01_-16ms
Finished F01_-12ms
Finished F01_-8ms
Finished F01_-4ms
Finished F01_0ms
Finished F01_4ms
Finished F01_8ms
Finished F01_12ms
Finished F01_16ms
Finished F01_20ms
Finished F03_-20ms
Finished F03_-16ms
Finished F03_-12ms
Finished F03_-8ms
Finished F03_-4ms
Finished F03_0ms
Finished F03_4ms
Finished F03_8ms
Finished F03_12ms
Finished F03_16ms
Finished F03_20ms
Finished F05_-20ms
Finished F05_-16ms
Finished F05_-12ms
Finished F05_-8ms
Finished F05_-4ms
Finished F05_0ms
Finished F05_4ms
Finished F05_8ms
Finished F05_12ms
Finished F05_16ms
Finished F05_20ms
Finished F07_-20ms
Finished F07_-16ms
Finished F07_-12ms
Finished F07_-8ms
Finished F07_-4ms
Finished F07_0ms
Finished F07_4ms
Finished F07_8ms
Finished F07_12ms
Finished F07_16ms
Finished F07_20ms
Finished F11_-20ms
Finished F11_-16ms
Finished F11_-12ms
Finished F11_-8ms
Finished F11_-4ms
Finished F11_0ms
Finished F11_4ms
Finished F11_8ms
Finished F11_12ms
Finished F11_16ms
Finished F11_20ms
Finished F

In [29]:
#Get all the times into one csv
#NO MORE CHANGE
dfs = []

for file in os.listdir(output_folder):
    if file.endswith(".csv") and file != "Wav2vec_OG.csv" and not file.startswith("combined"):
        #Get path of each file
        file_path = os.path.join(output_folder, file)
       
        # Read CSV
        df = pd.read_csv(file_path)

        #Get the name without the extension
        base = os.path.splitext(file)[0]

        #Split each word in the name
        parts = base.split("_")
        #print(parts)

        # The timing value is in index 2
        timing = parts[2]   
    
         # Rename columns to timing_columnname
        df.columns = [f"{timing}_{col}" for col in df.columns]

        # Store
        dfs.append(df)

#All the shifted csv appending
combined_df = pd.concat(dfs, axis=1)

# Read the non-shifted
df_wav2vec = pd.read_csv("Wav2Vec_Shifted/Wav2Vec_OG.csv")

# Reset index
df_wav2vec.reset_index(drop=True, inplace=True)

#combine them into one
full_df = pd.concat([combined_df, df_wav2vec], axis = 1)

full_df.to_csv("Wav2Vec_Shifted/combined.csv", index=False)

print("Done:", full_df.shape)

Done: (165, 123)


In [73]:
#REORDER
#Not needed because of the magical realignment function

full =  pd.read_csv("Wav2Vec_Shifted/combined.csv")

res = full.columns.tolist()  # Convert to list
#print(res)  # This will show ALL columns

# Custom sorting function for negative and positive numbers
def sort_key(col_name):
    # Handle the special case columns without numbers
    if col_name in ['word', 'start', 'end']:
        return (float('inf'), {'word': 0, 'start': 1, 'end': 2}[col_name])
    
    # Extract number and suffix
    match = re.match(r'(-?\d+)ms_(\w+)', col_name)  # Note: -? to handle negative numbers
    if match:
        num = int(match.group(1))  # This will be negative for -20, -19, etc.
        suffix = match.group(2)
        # Priority: word < start < end
        suffix_priority = {'word': 0, 'start': 1, 'end': 2}
        return (num, suffix_priority.get(suffix, 3))
    
    # Fallback
    return (float('inf'), col_name)

# Sort columns
sorted_cols = sorted(full.columns.tolist(), key=sort_key)
full = full[sorted_cols]

full.to_csv("Wav2Vec_Shifted/combined_sorted.csv", index=False)

full

#Now perform row alignment

,-20ms_word,-20ms_start,-20ms_end,-19ms_word,-19ms_start,-19ms_end,-18ms_word,-18ms_start,-18ms_end,-17ms_word,...,18ms_end,19ms_word,19ms_start,19ms_end,20ms_word,20ms_start,20ms_end,word,start,end
0,in,0.040,0.080,in,0.040,0.080,in,0.040,0.080,in,...,0.120,in,0.060,0.120,in,0.060,0.120,in,0.040,0.100
1,the,0.160,0.220,the,0.160,0.220,the,0.160,0.220,the,...,0.260,the,0.200,0.260,the,0.200,0.260,the,0.180,0.240
2,bosom,0.300,0.620,bosom,0.300,0.620,bosom,0.300,0.620,bosom,...,0.660,bosom,0.340,0.660,bosom,0.340,0.660,bosom,0.320,0.640
3,of,0.760,0.800,of,0.760,0.800,of,0.760,0.800,of,...,0.840,of,0.780,0.840,of,0.800,0.840,of,0.780,0.820
4,one,0.920,1.000,one,0.920,1.000,one,0.920,1.000,one,...,1.040,one,0.960,1.040,one,0.960,1.040,one,0.940,1.020
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,the,61.014,61.054,the,61.014,61.054,the,61.014,61.054,the,...,61.374,the,61.054,61.094,the,61.054,61.094,the,61.034,61.074
161,name,61.134,61.334,name,61.134,61.334,name,61.154,61.334,name,...,61.494,name,61.174,61.374,name,61.174,61.374,name,61.154,61.354
162,of,61.414,61.454,of,61.414,61.454,of,61.414,61.454,of,...,61.954,of,61.454,61.494,of,61.454,61.494,of,61.434,61.474
163,sleepy,61.614,61.914,sleepy,61.614,61.914,sleepy,61.614,61.914,sleepy,...,62.394,sleepy,61.654,61.954,sleepy,61.654,61.954,sleepy,61.634,61.934


In [69]:
#Now split into two dfs
#This is assuming the column without shift is at the end

idx_halfway = round(len(shift_ms)/2) * 3

neg_split = full.iloc[:,:idx_halfway]
neg_split = pd.concat([neg_split, no_shift],axis = 1)

neg_split.to_csv("Wav2Vec_Shifted/combined_neg.csv", index=False)

pos_split = full.iloc[:, idx_halfway:]
pos_split.to_csv("Wav2Vec_Shifted/combined_pos.csv", index=False)

In [18]:
# Decode the *entire* sequence of predicted IDs into a transcription
phoneme_transcription = phoneme_processor.decode(p_predicted_ids[0], group_tokens = False)
#print (transcription)

In [30]:
print("processing words")
words, word_start_times, word_end_times = decoding_to_timings(transcription, \
        predicted_ids, input_values, processor, target_sample_rate)

word_data = []

for w, s, e in zip(words, word_start_times, word_end_times):
    word_data.append({
        "word": w,
        "start": s,
        "end": e
    })

df = pd.DataFrame(word_data)
print(df.head())

processing words
  word start end
0    w     s   e
1    o     t   n
2    r     a   d


In [15]:
# RIGHT NOW DO NOT WORRY ABOUT ANYTHING BELOW
#Get timings for all phonemes at once
symbols, start_times, end_times = decode_to_time(
    transcription,
    p_predicted_ids,
    p_input_values,
    phoneme_processor,
    target_sample_rate
)
